# Assignment 3

## Interactive Dashboard for Video Game Sales Analysis

## Jelisa Lugo
### jnlugo@umich.edu

#### **Dataset:** vgsales_enriched.csv

#### **Source:** Raj Aditya Nag. *Video Game Sales Data with Other Data*. Kaggle.

#### **Visualization Library:** Plotly

________________________________________________________________________________________________________________________________________________________

## Introduction

In this project, I use the enriched version of the Video Game Sales dataset, called Video Game Sales Data with Other Data, obtained from Kaggle. I explore how game sales vary across genres, platforms, publishers, and release years. Before creating the dashboard, I cleaned the data and performed exploratory data analysis (EDA) to better understand the patterns in the dataset. I then used those findings to build an interactive dashboard in Plotly that makes the data easier to explore.

## Import Libraries

In [1]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import ipywidgets as widgets
from IPython.display import display
from ipywidgets import interact

In [2]:
import plotly.io as pio

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    pio.renderers.default = "colab"
    renderer = "colab"
except ImportError:
    renderer = "notebook"


## Load Dataset

In [3]:
try:
    # Local file (Jupyter Notebook / Vocareum)
    df = pd.read_csv("vgsales_enriched.csv")
except FileNotFoundError:
    # GitHub file (Google Colab)
    df = pd.read_csv("https://raw.githubusercontent.com/jnlugo/SIADS-5-Assignment-3-/refs/heads/main/vgsales_enriched.csv")

## Initial Data Inspection

In [4]:
df.shape

(16719, 19)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16719 entries, 0 to 16718
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Name            16717 non-null  object 
 1   Franchise_Name  16717 non-null  object 
 2   Is_Franchise    16719 non-null  int64  
 3   Platform        16719 non-null  object 
 4   Year            16450 non-null  float64
 5   Release_Month   16021 non-null  float64
 6   Genre           16717 non-null  object 
 7   Publisher       16665 non-null  object 
 8   Developer       10096 non-null  object 
 9   Rating          9950 non-null   object 
 10  NA_Sales        16719 non-null  float64
 11  EU_Sales        16719 non-null  float64
 12  JP_Sales        16719 non-null  float64
 13  Other_Sales     16719 non-null  float64
 14  Global_Sales    16719 non-null  float64
 15  Critic_Score    8137 non-null   float64
 16  Critic_Count    8137 non-null   float64
 17  User_Score      10015 non-null 

The dataset contains 16,719 rows and 19 columns. The variables include both numeric and categorical data related to video game sales, game information, and review scores.

In [6]:
df.head()

,Name,Franchise_Name,Is_Franchise,Platform,Year,Release_Month,Genre,Publisher,Developer,Rating,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Score,User_Count
0,Wii Sports,Wii Sports,0,Wii,2006.0,11.0,Sports,Nintendo,Nintendo,E,41.36,28.96,3.77,8.45,82.53,76.0,51.0,8,322.0
1,Super Mario Bros.,Super Mario Bros.,1,NES,1985.0,10.0,Platform,Nintendo,NaN,NaN,29.08,3.58,6.81,0.77,40.24,NaN,NaN,NaN,NaN
2,Mario Kart Wii,Mario Kart Wii,0,Wii,2008.0,4.0,Racing,Nintendo,Nintendo,E,15.68,12.76,3.79,3.29,35.52,82.0,73.0,8.3,709.0
3,Wii Sports Resort,Wii Sports Resort,0,Wii,2009.0,7.0,Sports,Nintendo,Nintendo,E,15.61,10.93,3.28,2.95,32.77,80.0,73.0,8,192.0
4,Pokemon Red/Pokemon Blue,Pokemon Red/Pokemon Blue,0,GB,1996.0,NaN,Role-Playing,Nintendo,NaN,NaN,11.27,8.89,10.22,1.00,31.37,NaN,NaN,NaN,NaN


## Data Quality Inspection

Before cleaning the data, I checked for missing values, duplicate records, summary statistics, and variable types to better understand the overall quality of the dataset.

In [7]:
df.isnull().sum()

,0
Name,2
Franchise_Name,2
Is_Franchise,0
Platform,0
Year,269
Release_Month,698
Genre,2
Publisher,54
Developer,6623
Rating,6769


While looking through the dataset, I noticed some missing values, mostly in the columns realted to reviews of the games. Because they weren't going to impact my analysis I was doing, I decided to leave them as they were.

In [8]:
df.describe()

,Is_Franchise,Year,Release_Month,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Count
count,16719.000000,16450.000000,16021.000000,16719.000000,16719.000000,16719.000000,16719.000000,16719.000000,8137.000000,8137.000000,7590.000000
mean,0.682397,2006.487356,7.358529,0.263330,0.145025,0.077602,0.047332,0.533543,68.967679,26.360821,162.229908
std,0.465558,5.878995,3.409766,0.813514,0.503283,0.308818,0.186710,1.547935,13.938165,18.980495,561.282326
min,0.000000,1980.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.010000,13.000000,3.000000,4.000000
25%,0.000000,2003.000000,4.000000,0.000000,0.000000,0.000000,0.000000,0.060000,60.000000,12.000000,10.000000
50%,1.000000,2007.000000,8.000000,0.080000,0.020000,0.000000,0.010000,0.170000,71.000000,21.000000,24.000000
75%,1.000000,2010.000000,10.000000,0.240000,0.110000,0.040000,0.030000,0.470000,79.000000,36.000000,81.000000
max,1.000000,2020.000000,12.000000,41.360000,28.960000,10.220000,10.570000,82.530000,98.000000,113.000000,10665.000000


In [9]:
df.duplicated().sum()

np.int64(0)

No duplicate observations were found.

## Data Cleaning

After reviewing the dataset, I found that most of the data was already clean. The main issue was the User_Score column, which contained text values that needed to be converted into numeric values before they could be analyzed.

In [10]:
df["User_Score"] = df["User_Score"].replace("tbd", np.nan)

In [11]:
df["User_Score"] = pd.to_numeric(df["User_Score"])

In [12]:
df["User_Score"].dtype

dtype('float64')

The User_Score column contained the value "tbd" (to be determined), which prevented the column from being treated as numeric. I replaced those values with NaN and then converted the column to a numeric data type. Since those scores were unavailable rather than wrong, I left the missing values as NaN.

In [13]:
df["Year"].unique()[:20]

array([2006., 1985., 2008., 2009., 1996., 1989., 1984., 2005., 1999.,
       2007., 2010., 2013., 2004., 1990., 1988., 2002., 2001., 2011.,
       1998., 2015.])

In [14]:
df["Year"].dtype

dtype('float64')

I also checked the Year column to see if it needed additional cleaning. It is stored as a float and every value represents a whole calendar year. I left the column unchanged.

## Exploratory Data Analysis (EDA)

Before building the dashboard, I wanted to explore the data to understand the main trends and relationships. This analysis helps me decide which visualizations make the most sense to include in the dashboard.

### Which genres generate the highest global sales?

I started by looking at total global sales for each genre to see which types of games sold the most.

In [15]:
genre_sales = (df.groupby("Genre")["Global_Sales"].sum().sort_values(ascending=False).reset_index())
genre_sales

,Genre,Global_Sales
0,Action,1745.27
1,Sports,1332.00
2,Shooter,1052.94
3,Role-Playing,934.40
4,Platform,828.08
5,Misc,803.18
6,Racing,728.90
7,Fighting,447.48
8,Simulation,390.42
9,Puzzle,243.02


In [16]:
# Created an plotly interactive bar chart to compare total global sales by genre so it would easier to create the dashboard later on.
fig = px.bar(
    genre_sales,
    x="Genre",
    y="Global_Sales",
    title="Global Video Game Sales by Genre"
)

fig.show()

### Genre Sales Analysis

Action games had the highest total global sales, followed by Sports and Shooter games. I was a little surprised to see Adventure games much lower on the list because they are a popular genre. One possible reason is that many action-adventure games are classified as Action instead of Adventure.

In [17]:
platform_sales = (df.groupby("Platform")["Global_Sales"].sum().sort_values(ascending=True).reset_index())
platform_sales.head(10)

,Platform,Global_Sales
0,PCFX,0.03
1,GG,0.04
2,3DO,0.10
3,TG16,0.16
4,WS,1.42
5,NG,1.44
6,SCD,1.87
7,DC,15.97
8,GEN,30.78
9,SAT,33.59


In [18]:
# Created an interactive horizontal bar chart to compare total global sales by platform.
fig = px.bar(
    platform_sales,
    x="Global_Sales",
    y="Platform",
    orientation="h",
    title="Global Video Game Sales by Platform"
)
fig.show()

### Platform Sales Analysis

The PlayStation 2 generated the highest total global sales, followed by the Xbox 360 and PlayStation 3. I was also surprised that the PlayStation 4 and PC ranked lower than expected. Since this dataset contains historical sales, older platforms like the PlayStation 2 had many more years to build large game libraries and accumulate sales.

In [19]:
year_sales = (df.groupby("Year")["Global_Sales"].sum().reset_index())
year_sales

,Year,Global_Sales
0,1980.0,11.38
1,1981.0,35.77
2,1982.0,28.86
3,1983.0,16.79
4,1984.0,50.36
5,1985.0,53.94
6,1986.0,37.07
7,1987.0,21.74
8,1988.0,47.22
9,1989.0,73.45


In [20]:
# Created an interactive line chart to show global sales over time.
fig = px.line(
    year_sales,
    x="Year",
    y="Global_Sales",
    title="Global Video Game Sales Over Time"
)

fig.show()

### Sales Trends Over Time

Sales continued to grow through the late 1990s and early 2000s before reaching their highest point in 2008. After that, total sales started to decline. I also noticed that the last few years in the dataset have very low sales, but I believe it's due to incomplete data rather than low sales.

In [21]:
score_sales = (df[["Critic_Score", "Global_Sales"]].dropna().corr())
score_sales

,Critic_Score,Global_Sales
Critic_Score,1.000000,0.245471
Global_Sales,0.245471,1.000000


In [22]:
df[["User_Score", "Global_Sales"]].corr()

,User_Score,Global_Sales
User_Score,1.000000,0.088139
Global_Sales,0.088139,1.000000


### Review Scores and Global Sales

I also wanted to see if higher review scores were connected to higher game sales. The Critic scores had a weak positive relationship with global sales (r = 0.245), while user scores had an even weaker relationship (r = 0.088). Based on these findings, review scores alone don't have a strong impact on how well a game sells, there are probably other factors that I cannot see that cause a bigger impact.

In [23]:
# keeping only the data that has both critic scores and global sales
scatter_data = (df[["Critic_Score", "Global_Sales"]].dropna())

In [24]:
# Created an interactive scatter plot to compare critic scores and global sales.
fig = px.scatter(
    scatter_data,
    x="Critic_Score",
    y="Global_Sales",
    title="Critic Scores vs. Global Video Game Sales"
)

fig.show()

In [25]:
publisher_sales = (df.groupby("Publisher")["Global_Sales"].sum().sort_values(ascending=False))
publisher_sales.head()

,Global_Sales
Publisher,
Nintendo,1788.81
Electronic Arts,1116.96
Activision,731.16
Sony Computer Entertainment,606.48
Ubisoft,471.61


### Publisher Sales Analysis

Nintendo had the highest total global sales in the dataset, followed by Electronic Arts and Activision. I was't too surprised because Nintendo has some of the biggest game franchises, like Mario and Pokémon. Electronic Arts and Activision also ranked near the top because of their popular sports and action games.

#### How do video game genres perform across different regions?

In [26]:
region_sales = (
    df.groupby("Genre")[[
        "NA_Sales",
        "EU_Sales",
        "JP_Sales",
        "Other_Sales"
    ]]
    .sum()
)

region_sales

,NA_Sales,EU_Sales,JP_Sales,Other_Sales
Genre,,,,
Action,879.01,519.13,161.44,184.60
Adventure,105.26,63.54,52.30,16.49
Fighting,223.36,100.33,87.48,36.36
Misc,407.27,212.74,108.11,74.39
Platform,445.50,200.35,130.83,51.09
Puzzle,122.87,50.01,57.31,12.38
Racing,359.35,236.51,56.71,76.10
Role-Playing,330.81,188.71,355.46,59.63
Shooter,592.24,317.34,38.76,104.11


Before creating the heatmap, I grouped the data by genre and calculated the total sales for each region. This made it easier to compare how genres performed across North America, Europe, Japan, and other regions.

In [27]:
fig = px.imshow(
    region_sales,
    text_auto=True,
    color_continuous_scale="Blues",
    title="Video Game Sales by Genre and Region"
)

# I Renamed the region labels so they are easier to read.
fig.update_xaxes(
    ticktext=["North America", "Europe", "Japan", "Other"],
    tickvals=[0, 1, 2, 3]
)

# I Increased the figure size so the heatmap is easier to view thna using the auto since it was very small with the auto layout.
fig.update_layout(
    width=900,
    height=600
)

fig.show()

The heatmap made it easier to compare how each genre performed across the different regions. I noticed that Action games had the highest sales in both North America and Europe. I expected Role-Playing games to perform well in Japan because many popular Japanese games and franchises are built around role-playing mechanics. Shooter games also had much stronger sales in North America than in Japan. Overall, North America had the highest sales across most genres, with Europe generally ranking second.

#### How are global sales distributed across publishers and genres?

In [28]:
treemap_data = (df[['Publisher', 'Genre', 'Global_Sales']].dropna())
treemap_data

,Publisher,Genre,Global_Sales
0,Nintendo,Sports,82.53
1,Nintendo,Platform,40.24
2,Nintendo,Racing,35.52
3,Nintendo,Sports,32.77
4,Nintendo,Role-Playing,31.37
...,...,...,...
16714,Tecmo Koei,Action,0.01
16715,Codemasters,Sports,0.01
16716,Idea Factory,Adventure,0.01
16717,Wanadoo,Platform,0.01


The table shows each game's publisher, genres, and global sales. I removed any missing values so the treemap would accuratly group games by publisher and genre with including incomplte data.

In [29]:
#Created an interactive treemap to show global sales by publisher and genre
fig = px.treemap(treemap_data,
                 path = ['Publisher','Genre'],
                 values= 'Global_Sales',
                 title= 'Global Video Game Sales by Publisher and Genre')
fig.show()


I found that Nintendo had the largest share of global sales, followed by Electronic Arts and Activision. The treemap also helped me see which genres contributed the most sales for each publisher.

## EDA Summary

Overall, the analysis gave me a better understanding of the dataset before building the dashboard. I found that Action games had the highest total sales, the PlayStation 2 was the top-selling platform, Nintendo was the leading publisher, and game sales peaked around 2008. I also found that review scores had only a weak relationship with global sales. These results helped me decide which charts and filters to include in the interactive dashboard.

## Visualization Library

For this project, I chose Plotly because I wanted to build an interactive dashboard instead of using static charts. Plotly is a free, open-source data visualization library developed by Plotly Technologies that works well with Python and Jupyter notebooks. It supports interactive features like zooming, hovering, panning, and filtering, making it easier for users to explore the data and identify patterns.

To install Plotly, you can use:

python pip install plotly

I chose Plotly because my dashboard uses a few different visualization types, including bar charts, a line chart, a scatter plot, a heatmap, and a treemap. Plotly supports all of these in one library and allows them to be interactive, which makes it a good fit for exploring different parts of the dataset together.

Another reason I chose Plotly is that it integrates well with Jupyter notebooks, which made it easy to build and demonstrate the dashboard within this assignment. Compared to libraries that mainly create static figures, Plotly focuses on interactive visualizations, making it a good choice for dashboards where users can explore the data on their own.

One limitation of Plotly is that creating dashboards with multiple linked charts requires more code than creating individual static graphs. For larger web applications, Plotly also provides Dash, which can be used to build more advanced interactive dashboards.

## Visualization Techniques

The dashboard uses several different visualization types because each one answers a different question about the data.

#### Bar Chart
I used bar charts to compare total sales across genres, platforms, and publishers because they make it easy to see which categories performed the best.

#### Scatter Plot
I included a scatter plot to see whether games with higher review scores also had higher sales. It helped me quickly see that the relationship between review scores and sales was fairly weak.

#### Heatmap

I used a heatmap to compare video game sales across different genres and regions. The differences in shading make it easier to see which genres perform stronger in North America, Europe, Japan, and other regions and to identify regional sales patterns.

#### Treemap

I used a treemap to explore how video game sales are distributed across publishers and genres. The size of each section represents the amount of sales, which makes it easier to see which publishers and genres make up the largest parts of the market.

#### Dashboard Design
These visualizations each show a different part of the video game sales story. The bar charts compare categories, the line chart shows changes over time, the scatter plot explores the relationship between reviews and sales, the heatmap compares regional patterns, and the treemap shows how sales are distributed across larger groups. In the final dashboard, interactive filters allow users to explore the data while multiple visualizations update together.

### Interactive Dashboard

After creating the individual visualizations, I combined them into an interactive dashboard using Plotly and ipywidgets. The dashboard allows users to filter the dataset and update multiple visualizations at the same time, making it easier to explore sales trends across different genres, publishers, platforms, and years.

In [30]:
# Created an interactive dashboard.

print("Interactive Video Game Sales Dashboard")
print("Use the filters below to update all four visualizations.")


def create_dashboard(filtered_df):

    # Calculate total global sales by genre.
    genre_sales = (filtered_df.groupby("Genre")["Global_Sales"].sum().sort_values(ascending=False).reset_index())

    # Calculated total global sales by year.
    year_sales = (filtered_df.groupby("Year")["Global_Sales"].sum().reset_index())

    # Removed missing values for scatter plot.
    scatter_df = filtered_df.dropna(subset=["Critic_Score", "Global_Sales"])

    # Removed missing values for treemap.
    treemap_df = filtered_df.dropna(subset=["Publisher", "Genre", "Global_Sales"])

    # Created a 2 x 2 dashboard.
    db = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=(
            "Global Video Game Sales by Genre",
            "Global Video Game Sales Over Time",
            "Critic Score vs. Global Sales",
            "Global Sales by Publisher and Genre"
        ),
        specs=[
            [{"type": "bar"}, {"type": "scatter"}],
            [{"type": "scatter"}, {"type": "domain"}]
        ]
    )

    # Created bar chart.
    bar_fig = px.bar(
        genre_sales,
        x="Genre",
        y="Global_Sales"
    )

    for chart in bar_fig.data:
        db.add_trace(chart, row=1, col=1)

    # Created line chart.
    line_fig = px.line(
        year_sales,
        x="Year",
        y="Global_Sales"
    )

    for chart in line_fig.data:
        db.add_trace(chart, row=1, col=2)

    # Create scatter plot.
    scatter_fig = px.scatter(
        scatter_df,
        x="Critic_Score",
        y="Global_Sales",
        hover_data=["Name"]
    )

    for chart in scatter_fig.data:
        db.add_trace(chart, row=2, col=1)

    # Created treemap.
    tree_fig = px.treemap(
        treemap_df,
        path=["Publisher", "Genre"],
        values="Global_Sales"
    )

    for chart in tree_fig.data:
        db.add_trace(chart, row=2, col=2)

    # Updated the dashboard layout
    db.update_layout(
        title="Interactive Video Game Sales Dashboard",
        height=900,
        width=1100,
        showlegend=False
    )

    return db


@interact(
    genre=["All"] + sorted(df["Genre"].dropna().unique().tolist()),
    publisher=["All"] + sorted(df["Publisher"].dropna().unique().tolist()),
    year=widgets.IntRangeSlider(
        value=[int(df["Year"].min()), int(df["Year"].max())],
        min=int(df["Year"].min()),
        max=int(df["Year"].max()),
        step=1,
        description="Year:"
    )
)

def update_dashboard(genre, publisher, year):

    filtered_df = df.copy()

    # Filtered the data by genre
    if genre != "All":filtered_df = filtered_df[filtered_df["Genre"] == genre]

    # Filtered the data by publisher
    if publisher != "All":filtered_df = filtered_df[filtered_df["Publisher"] == publisher]

    # Filtered the data by release year
    filtered_df = filtered_df[(filtered_df["Year"] >= year[0]) & (filtered_df["Year"] <= year[1])]

    create_dashboard(filtered_df).show(renderer=renderer)

Interactive Video Game Sales Dashboard
Use the filters below to update all four visualizations.


interactive(children=(Dropdown(description='genre', options=('All', 'Action', 'Adventure', 'Fighting', 'Misc',…

The completed dashboard combines my four visualizations into a single interactive display. Users can filter the data by genre, publisher, and release year. Each filter updates all four visualizations simultaneously, making it easier to compare trends across multiple perspectives.

## Deployment

The completed dashboard is hosted publicly so it can be viewed without installing any software. The source code is available in my GitHub repository, and the dashboard can be accessed using the link below.

GitHub Repository:https://github.com/jnlugo/SIADS-5-Assignment-3-

Dashboard:

## Conclusion
In this project, I cleaned and explored a video game sales dataset before creating an interactive dashboard with Plotly. The dashboard combines multiple visualization types to help users compare genres, platforms, publishers, review scores, and sales trends over time. Building the dashboard gave me experience creating interactive visualizations and showed how combining several charts can tell a more complete story than using a single graph. Overall, this project demonstrated how interactive dashboards can make large datasets easier to explore by allowing users to filter the data and compare multiple visualizations at the same time.

## Troubleshooting

One issue I encountered was that the User_Score column was stored as text because it contained "tbd" values. I replaced those values with NaN before converting the column to a numeric data type.

Another challenge was deciding how to handle missing review scores. Since those values were not required for every visualization, I left them as missing instead of filling them in.

Another was while building the dashboard, I originally displayed each visualization separately. I later combined the charts into a single interactive dashboard using Plotly's make_subplots() so that all visualizations appeared together on one screen.